# 数据准备

In [1]:
import sklearn.datasets as datasets
import sklearn.model_selection as model_selection
import pandas as pd

In [2]:
iris = datasets.load_iris(
    as_frame=True
)
x,y = iris.data, iris.target

print(
    type(x), type(y)
)

<class 'pandas.DataFrame'> <class 'pandas.Series'>


## 数据划分

### 数据划分策略

In [7]:
import sklearn.datasets as datasets
import sklearn.model_selection as model_selection
import sklearn.preprocessing as preprocessing
import sklearn.linear_model as linear_model
import sklearn.neighbors as neighbors
import sklearn.metrics as metrics
import sklearn.pipeline as pipeline

X,y = datasets.load_iris( return_X_y=True )
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)
X_train, X_val, y_train, y_val = model_selection.train_test_split(
    X_train, y_train,
    test_size = 0.25,
    random_state = 42,
    stratify = y_train
)

for k in [1, 3, 5, 7, 9, 11]:
    pipe = pipeline.Pipeline([
        ("scaler", preprocessing.StandardScaler()),
        ("model", neighbors.KNeighborsClassifier(n_neighbors=k))
    ])

    pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    val_acc = metrics.accuracy_score(y_val, y_val_pred)

    print(f"k={k}, validation_acc = {val_acc}")

k=1, validation_acc = 0.9
k=3, validation_acc = 0.9333333333333333
k=5, validation_acc = 0.9333333333333333
k=7, validation_acc = 0.9666666666666667
k=9, validation_acc = 0.9666666666666667
k=11, validation_acc = 0.9666666666666667


##### Cross Validation

In [18]:
X,y = datasets.load_breast_cancer( return_X_y = True )

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X,y,
    test_size = 0.25,
    random_state = 42,
    stratify = y
)

pipe = pipeline.Pipeline([
    ("scaler", preprocessing.StandardScaler()),
    ("model", linear_model.LogisticRegression(max_iter=1000, random_state=42))
])

scores = model_selection.cross_val_score(
    estimator = pipe,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = "accuracy"
)

print(
    f"每一折分数: {scores}",
    f"平均分数: {scores.mean()}",
    f"分数标准差: {scores.std()}",
    sep = "\n-----\n"
)

每一折分数: [0.97674419 0.97647059 0.95294118 0.96470588 0.97647059]
-----
平均分数: 0.969466484268126
-----
分数标准差: 0.009453348785897427


In [25]:
result = model_selection.cross_validate(
    pipe, X_train, y_train,
    cv = 5,
    scoring = ["accuracy", "precision", "recall", "f1"],
    return_train_score = True
)

result

{'fit_time': array([0.00600553, 0.00544763, 0.00650764, 0.00458097, 0.00500464]),
 'score_time': array([0.00441527, 0.00500774, 0.00445294, 0.00400209, 0.00300527]),
 'test_accuracy': array([0.97674419, 0.97647059, 0.95294118, 0.96470588, 0.97647059]),
 'train_accuracy': array([0.98823529, 0.98533724, 0.98826979, 0.98826979, 0.98826979]),
 'test_precision': array([0.96428571, 0.98113208, 0.92982456, 0.98076923, 0.98148148]),
 'train_precision': array([0.98604651, 0.98156682, 0.98611111, 0.98165138, 0.98604651]),
 'test_recall': array([1.        , 0.98113208, 1.        , 0.96226415, 0.98148148]),
 'train_recall': array([0.99530516, 0.9953271 , 0.9953271 , 1.        , 0.99530516]),
 'test_f1': array([0.98181818, 0.98113208, 0.96363636, 0.97142857, 0.98148148]),
 'train_f1': array([0.99065421, 0.98839907, 0.99069767, 0.99074074, 0.99065421])}

## 数据处理

### 文本编码

In [3]:
import sklearn.preprocessing as preprocessing

In [4]:
X = [
    ["北京"],
    ["上海"],
    ["广州"],
    ["北京"]
]

encoder = preprocessing.OneHotEncoder()
X_encoded = encoder.fit_transform(X)

print(X_encoded, encoder.categories_, sep = "\n-----\n")
X_encoded.toarray()

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (4, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 0)	1.0
  (2, 2)	1.0
  (3, 1)	1.0
-----
[array(['上海', '北京', '广州'], dtype=object)]


array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [5]:
X = [
    ["北京", "男"],
    ["上海", "女"],
    ["广州", "女"],
    ["北京", "男"]
]

encoder = preprocessing.OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_encoded = encoder.fit_transform(X)

print(
    X_encoded,
    encoder.categories_,
    sep = "\n-----\n"
)

[[0. 1. 0. 0. 1.]
 [1. 0. 0. 1. 0.]
 [0. 0. 1. 1. 0.]
 [0. 1. 0. 0. 1.]]
-----
[array(['上海', '北京', '广州'], dtype=object), array(['女', '男'], dtype=object)]


#### Ordinal Encoding

In [6]:
X = [
    ["低"],
    ["中"],
    ["高"],
    ["中"]
]

encoder = preprocessing.OrdinalEncoder()
print( encoder.fit_transform(X) )

encoder2 = preprocessing.OrdinalEncoder(categories=[["低", "中", "高"]])
print( encoder2.fit_transform(X) )

[[1.]
 [0.]
 [2.]
 [0.]]
[[0.]
 [1.]
 [2.]
 [1.]]


#### Label Encoding

In [7]:
y = ["setosa", "versicolor", "virginica", "setosa"]
encoder = preprocessing.LabelEncoder()

y_encoded = encoder.fit_transform(y)

print(y_encoded, encoder.classes_)

[0 1 2 0] ['setosa' 'versicolor' 'virginica']


### 缺失处理

In [8]:
import numpy as np
import sklearn.impute as impute

In [9]:
X = [
    [20, 5000],
    [35, np.nan],
    [np.nan, 8000],
    [42, 20000],
]

imputer = impute.SimpleImputer(
    strategy="mean"
)

X_filled = imputer.fit_transform(X)
print(
    X, X_filled,
    sep = "\n-----\n"
)

[[20, 5000], [35, nan], [nan, 8000], [42, 20000]]
-----
[[   20.          5000.        ]
 [   35.         11000.        ]
 [   32.33333333  8000.        ]
 [   42.         20000.        ]]


## 特征处理

### 特征构造

In [10]:
X = np.array([
    [2, 3],
    [4, 5]
])

poly = preprocessing.PolynomialFeatures(
    degree = 2
)

X_poly = poly.fit_transform(X)

print(
    X, X_poly,
    sep = "\n-----\n"
)

[[2 3]
 [4 5]]
-----
[[ 1.  2.  3.  4.  6.  9.]
 [ 1.  4.  5. 16. 20. 25.]]


### 特征选择

In [11]:
import sklearn.feature_selection as feature_selection

#### VarianceThreshold

In [12]:
X = np.array([
    [1, 2, 0],
    [1, 3, 1],
    [1, 4, 0],
    [1, 5, 1]
])

selector = feature_selection.VarianceThreshold()
X_new = selector.fit_transform(X)

print(X, X_new, sep="\n-----\n")

[[1 2 0]
 [1 3 1]
 [1 4 0]
 [1 5 1]]
-----
[[2 0]
 [3 1]
 [4 0]
 [5 1]]


#### SelectKBest

In [13]:
X,y = datasets.load_iris( return_X_y = True )
selector = feature_selection.SelectKBest( score_func = feature_selection.f_classif, k=2 )

X_new = selector.fit_transform(X, y)

print(X.shape, X_new.shape, X, X_new, sep="\n-----\n")


(150, 4)
-----
(150, 2)
-----
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 

#### SelectFromModel

In [14]:
import sklearn.ensemble as ensemble

X,y = datasets.load_iris(return_X_y=True)
estimator = ensemble.RandomForestClassifier(
    random_state=42
)

selector = feature_selection.SelectFromModel(
    estimator = estimator,
    threshold = "mean"
)

X_new = selector.fit_transform(X,y)

print(X.shape, X_new.shape)

(150, 4) (150, 2)


#### Recursive Feature Elimination

In [15]:
import sklearn.linear_model as linear_model

X, y = datasets.load_iris(return_X_y=True)
estimator = linear_model.LogisticRegression(
    max_iter = 1000
)

selector = feature_selection.RFE(
    estimator = estimator,
    n_features_to_select = 2
)

X_new = selector.fit_transform(X, y)
print(
    X.shape, X_new.shape, selector.support_, selector.ranking_,
    sep = "\n-----\n"
)

(150, 4)
-----
(150, 2)
-----
[False False  True  True]
-----
[3 2 1 1]


## 总结练习

In [16]:
import numpy as np
import pandas as pd

X = pd.DataFrame({
    "age": [20, 35, np.nan, 42, 50, 23, 31, 46, 52, 27, np.nan, 39],
    "income": [5000, np.nan, 8000, 20000, 30000, 6000, 12000, 22000, np.nan, 7500, 9000, 18000],
    "city": ["北京", "上海", "北京", np.nan, "广州", "上海", "北京", "广州", "深圳", "上海", np.nan, "北京"],
    "gender": ["男", np.nan, "女", "男", "女", "男", "女", "男", "女", "男", "女", np.nan]
})

y = [0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1]
X

,age,income,city,gender
0,20.0,5000.0,北京,男
1,35.0,NaN,上海,NaN
2,NaN,8000.0,北京,女
3,42.0,20000.0,NaN,男
4,50.0,30000.0,广州,女
5,23.0,6000.0,上海,男
6,31.0,12000.0,北京,女
7,46.0,22000.0,广州,男
8,52.0,NaN,深圳,女
9,27.0,7500.0,上海,男


In [17]:
import sklearn.model_selection as model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y,
    test_size = 0.25,
    random_state = 42,
    stratify = y
)

print(X_train, y_train, sep="\n---\n")

     age   income city gender
0   20.0   5000.0   北京      男
2    NaN   8000.0   北京      女
11  39.0  18000.0   北京    NaN
1   35.0      NaN   上海    NaN
5   23.0   6000.0   上海      男
7   46.0  22000.0   广州      男
4   50.0  30000.0   广州      女
10   NaN   9000.0  NaN      女
3   42.0  20000.0  NaN      男
---
[0, 0, 1, 1, 0, 1, 1, 0, 1]


In [18]:
# 数据处理
import sklearn.pipeline as pipeline
import sklearn.impute as impute
import sklearn.preprocessing as preprocessing

numeric_feature = ["age", "income"]
categorial_feature = ["city", "gender"]

numeric_pipeline = pipeline.Pipeline(
    steps = [
        ("imputer", impute.SimpleImputer(strategy="median")),
        ("scaler", preprocessing.StandardScaler())
    ]
)

categorial_pipeline = pipeline.Pipeline(
    steps = [
        ("imputer", impute.SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", preprocessing.OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [19]:
import sklearn.compose as compose

preprocessor = compose.ColumnTransformer(
    transformers = [
        ("num", numeric_pipeline, numeric_feature),
        ("cat", categorial_pipeline, categorial_feature)
    ]
)

In [20]:
# 特征处理
import sklearn.feature_selection as feature_selection

selector = feature_selection.SelectKBest(
    score_func = feature_selection.f_classif,
    k=5
)

In [21]:
import sklearn.linear_model as linear_model

model = linear_model.LogisticRegression(
    max_iter = 1000
)

In [22]:
model_pipeline = pipeline.Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("selector", selector),
        ("model", model)
    ]
)

In [23]:
model_pipeline.fit(X_train, y_train)
score = model_pipeline.score(X_test, y_test)

print(score)

1.0


# 数据建模

# 数据工程

### 通用工具

##### Column Transformer

In [24]:
import pandas as pd
import sklearn.compose as compose
import sklearn.preprocessing as preprocessing

In [25]:
X = pd.DataFrame({
    "age": [20, 35, 28, 42],
    "income": [5000, 12000, 8000, 20000],
    "city": ["北京", "上海", "广州", "北京"],
    "gender": ["男", "女", "女", "男"]
})

numerical_feature = ["age", "income"]
categorial_feature = ["city", "gender"]

preprocessor = compose.ColumnTransformer(
    [
        ("num", preprocessing.StandardScaler(), numerical_feature),
        ("cat", preprocessing.OneHotEncoder(), categorial_feature)
    ]
)

X_transformed = preprocessor.fit_transform(X)
print(
    X, X_transformed,
    type(X_transformed),
    sep = "\n-----\n"
)

   age  income city gender
0   20    5000   北京      男
1   35   12000   上海      女
2   28    8000   广州      女
3   42   20000   北京      男
-----
[[-1.37762274 -1.11028898  0.          1.          0.          0.
   1.        ]
 [ 0.45920758  0.13323468  1.          0.          0.          1.
   0.        ]
 [-0.3979799  -0.57735027  0.          0.          1.          1.
   0.        ]
 [ 1.31639507  1.55440457  0.          1.          0.          0.
   1.        ]]
-----
<class 'numpy.ndarray'>
